In [1]:
from langchain_chroma import Chroma
from pdf_chunk import text_spliter
from models import get_model , get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda


e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm= get_model()
embedding_model = get_embeddings()
chunks = text_spliter
chroma_vectorstore=Chroma.from_documents(chunks ,embedding_model )
chroma_retriever = chroma_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1771.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Prompt to generate multiple versions of the original query
multi_query_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Generate {n} different versions of the given question 
to retrieve relevant documents from a vector database.
Write each question on a new line. Do not number them.

Original Question: {question}
""")

In [4]:
def generate_queries(question: str, n: int = 5):
    chain = multi_query_prompt | llm | StrOutputParser()
    result = chain.invoke({"question": question, "n": n})
    queries = [q.strip() for q in result.strip().split("\n") if q.strip()]
    print(f"Generated {len(queries)} queries:")
    for q in queries:
        print(f"  → {q}")
    return queries

In [10]:
queries = generate_queries("what is current university of rohanta",5)

Generated 5 queries:
  → What is the current university associated with Rohanta?
  → What is the name of the university currently affiliated with Rohanta?
  → Which university is currently associated with Rohanta?
  → What is the current educational institution of Rohanta?
  → What university is currently linked to Rohanta?


In [11]:
def retrieve_for_all_queries(queries: list):
    all_results = {}
    for query in queries:
        docs = chroma_retriever.invoke(query)
        all_results[query] = docs
        print(f"\nQuery: '{query}' → {len(docs)} docs retrieved")
    return all_results

In [13]:
all_results = retrieve_for_all_queries(queries)


Query: 'What is the current university associated with Rohanta?' → 5 docs retrieved

Query: 'What is the name of the university currently affiliated with Rohanta?' → 5 docs retrieved

Query: 'Which university is currently associated with Rohanta?' → 5 docs retrieved

Query: 'What is the current educational institution of Rohanta?' → 5 docs retrieved

Query: 'What university is currently linked to Rohanta?' → 5 docs retrieved


In [14]:
def reciprocal_rank_fusion(results: dict, k: int = 60):
    """
    RRF Score = Σ 1 / (k + rank)
    Higher score = more relevant document
    """
    rrf_scores = {}  # doc_id → score
    doc_map = {}     # doc_id → actual document

    for query, docs in results.items():
        for rank, doc in enumerate(docs):
            doc_id = doc.page_content  # use content as unique ID
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0
                doc_map[doc_id] = doc
            # Add RRF score for this rank
            rrf_scores[doc_id] += 1 / (k + rank + 1)

    # Sort by score descending
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    print("\nRRF Scores:")
    for doc_id, score in sorted_docs[:5]:
        print(f"  Score: {score:.4f} | {doc_id[:80]}...")

    # Return sorted documents
    return [doc_map[doc_id] for doc_id, _ in sorted_docs]

In [16]:
fused_docs=reciprocal_rank_fusion(all_results)


RRF Scores:
  Score: 0.0817 | Before transitioning fully into industry, Rohanta taught undergraduate Computer ...
  Score: 0.0799 | Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: Ma...
  Score: 0.0776 | Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Seconda...
  Score: 0.0645 | IDENTITY AND CONTACT

Full Name: Rohanta Dinkar Bhamare
Title: AI Engineer
Locat...
  Score: 0.0469 | Rohanta uses LangChain to orchestrate LLM workflows. He uses components includin...


In [25]:
contents = []
for i in fused_docs:
    contents.append(i.page_content)

In [26]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [27]:
# First join contents into a single string
context_str = "\n\n".join(contents)

# Then pass directly
normal_chain = (
    {"context": RunnableLambda(lambda x: context_str), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [28]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

The current university of Rohanta is not explicitly mentioned in the provided context. However, based on the information given, we can infer that Rohanta is currently not affiliated with any university. 

The context mentions that Rohanta taught at Guru Gobind Singh Foundation from May 2019 to August 2022, indicating that he was an Assistant Professor at that institution during that period. However, there is no information about his current affiliation or university.
